# Notebook 19 — Kiểm chứng domain-wall ba host và development pilot

**Nhánh A; bước tiếp theo sau checkpoint Notebook18.** Chạy tuần tự trên kernel
nt409-khiem. Notebook này kiểm chứng mô hình, đọc dữ liệu đã khóa, đo controls và
chạy **12 development trials +2 functional smokes**. Có **6 dev probes +2 control probes**.
Tập test không được đưa vào QAOA. Không chọn p từ pilot, không có ngưỡng27/30,24/30.
Kết quả âm vẫn là kết quả hợp lệ. Notebook20 chưa được khởi chạy.

Lần đầu Restart Kernel rồi Run All. Nếu chia phiên, ghi lại đường dẫn được in ở
bước2 và đặt RESUME_DIR trước lần sau. Có thể chọn RUN_INSTANCES theo kích thước;
không đổi số shots, seed, penalty hay cap khi resume. Lượt completed không chạy lại.

## Bước1 — Môi trường và lựa chọn phiên chạy

Giữ nguyên cấu hình được cung cấp. RUN_INSTANCES có thể chỉ chứa a3_dev_n4 cho
phiên4task, rồi bổ sung n5/controls trong cùng folder. RETRY_FAILED mặc định False;
interrupted được recovery, các lỗi khác được giữ để review trước khi thử lại.

In [1]:
from pathlib import Path
import json, sys, platform
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / 'src/three_host_pilot.py').is_file())
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
import pandas as pd
import matplotlib.pyplot as plt
def display(value):
    print(value.to_string(index=False) if hasattr(value,"to_string") else value)
from src.three_host_benchmark import load_three_host_benchmark, measure_assignment
from src.three_host_pilot import (open_pilot, prepare_models, run_phase, result_rows,
                                  export_pilot, control_rows, verify_pilot)
from src.benchmark_a_campaign import save_json
RESUME_DIR = None  # Ví dụ: 'data/reference_results/phase4k_three_host_pilot/pilot_<id>'
RUN_INSTANCES = ['a3_dev_n4', 'a3_dev_n5', 'a3_one_host', 'a3_infeasible']
RETRY_FAILED = False
CONTEXT = load_three_host_benchmark(ROOT / 'data/benchmarks/nt409_three_host_v2')
print('Runtime:', platform.platform(), platform.python_version())
print('Dataset:', CONTEXT['manifest']['design_sha256'])
print('Cấu hình:', CONTEXT['protocol']['qaoa'])
from copy import deepcopy
CONTEXT=deepcopy(CONTEXT)
CONTEXT["protocol"]["qaoa"].update(shots=64,evaluation_shots=64,maxiter=8,max_seconds=600.)
CONTEXT["protocol"]["functional_smoke"]["maxiter"]=8
print("LINUX SOFTWARE VALIDATION: shots64/cap8; dataset unchanged; guards retained; not Windows evidence.")


Runtime: Linux-6.18.44-x86_64-with-glibc2.39 3.12.14
Dataset: 0df7ad1f683eb2ae696422d15833e55a73ff5a552e47cf6b33f965bd9fdb8aad
Cấu hình: {'depths': [1, 2, 3], 'optimization_seeds': [190, 191], 'evaluation_seeds': [100190, 100191], 'shots': 1024, 'evaluation_shots': 1024, 'maxiter': 200, 'max_seconds': 2400, 'hamiltonian_scaling': 'max_abs', 'cobyla_rhobeg': 1.0, 'cobyla_tol': 0.0001, 'initial_points_beta_then_gamma': {'1': [0.3, 1.0], '2': [0.3, 0.0, 1.0, 0.0], '3': [0.3, 0.0, 0.0, 1.0, 0.0, 0.0]}}
LINUX SOFTWARE VALIDATION: shots64/cap8; dataset unchanged; guards retained; not Windows evidence.


## Bước2 — Mở pilot mới hoặc tiếp tục pilot cũ

Manifest/hash/source/runtime được khóa để từ chối resume khi thay dữ liệu hoặc code.
Ghi lại RESUME_DIR được in ra. Đủ14 planned trials mới gọi pilot complete; test rows
không có trong planned_runs. Không xóa folder khi cần chia phiên.

In [2]:
FOLDER, CAMPAIGN = open_pilot(ROOT, CONTEXT, RESUME_DIR)
print('RESUME_DIR =', repr(FOLDER.relative_to(ROOT).as_posix()))
print('Planned pilot trials:', len(CAMPAIGN['planned_runs']))
display(pd.DataFrame(CONTEXT['manifest']['instances'])[['instance_id','role','num_tasks','num_qubits','optimum','uniform_batch_success']])
assert all(t['role'] != 'test' for t in CAMPAIGN['planned_runs'])

RESUME_DIR = 'data/reference_results/phase4k_three_host_pilot/pilot_20261012T032855209810Z'
Planned pilot trials: 14
  instance_id               role  num_tasks  num_qubits  optimum  uniform_batch_success
    a3_dev_n4        development          4          23      2.0               0.632300
 a3_test_n4_1               test          4          20      2.0               0.393543
 a3_test_n4_2               test          4          22      2.0               0.632300
 a3_test_n4_3               test          4          22      2.0               0.632300
    a3_dev_n5        development          5          25      2.0               0.632300
 a3_test_n5_1               test          5          24      2.0               0.221223
 a3_test_n5_2               test          5          24      2.0               0.393543
 a3_test_n5_3               test          5          25      2.0               0.632300
  a3_one_host functional_control          4          21      1.0                    NaN
a3_

## Bước3 — Kiểm chứng mô hình và nghiệm tham chiếu

Hai bit/task dùng00,10,11;01 không hợp lệ. Capacity/domain penalty4, y/slack giữ
nguyên. X mixer có thể tạo01; decoder ghi vi phạm, không sửa. Ground QUBO và
placement optimum được kiểm tra độc lập. Tính cả thời gian preprocessing/certificate.
Instance vô nghiệm: optimum/P_opt=null, không đánh giá là tối ưu0host.

In [3]:
MODELS = prepare_models(FOLDER, CONTEXT)
save_json(FOLDER / 'model_validation.json', MODELS)
display(pd.DataFrame([dict(instance_id=r['instance_id'],role=r['role'],qubits=r['num_qubits'],
    optimum=r['reference']['original_best_objective'],minimum_qubo=r['reference']['minimum_energy'],
    polynomial_identity=r['certificate']['polynomial_identity'],
    preprocessing_seconds=r['model_reference_certificate_seconds']) for r in MODELS]))
print('MODEL PASS; không chạy lượng tử trong bước này.')

  instance_id               role  qubits  optimum  minimum_qubo  polynomial_identity  preprocessing_seconds
    a3_dev_n4        development      23      2.0           2.0                 True               0.216741
 a3_test_n4_1               test      20      2.0           2.0                 True               0.187443
 a3_test_n4_2               test      22      2.0           2.0                 True               0.247663
 a3_test_n4_3               test      22      2.0           2.0                 True               0.264348
    a3_dev_n5        development      25      2.0           2.0                 True               0.288829
 a3_test_n5_1               test      24      2.0           2.0                 True               0.263821
 a3_test_n5_2               test      24      2.0           2.0                 True               0.268198
 a3_test_n5_3               test      25      2.0           2.0                 True               0.315156
  a3_one_host functional_con

## Bước4 — Đối chứng chính xác theo từng instance

Uniform encoded lấy bit domain-wall/y đều và marginal slack. Random assignment
chọn một trong3host/task rồi suy ra y. Hai phân bố khác nhau; lift hơn uniform
không có nghĩa hơn random placement. Batch probability chỉ áp IID cho controls.
Cột required_lift_for_80pct_batch là chẩn đoán, không phải tiêu chí nghiệm thu.

In [4]:
CONTROLS = control_rows(CONTEXT)
save_json(FOLDER / 'controls.json', CONTROLS)
display(pd.DataFrame(CONTROLS)[['instance_id','role','method','feasible_probability',
    'optimal_probability','optimal_batch_1024_probability','required_lift_for_80pct_batch']])
print('Random batch gần1 có thể hiển thị thành1 vì floating-point; không phải bảo đảm.')

  instance_id               role                        method  feasible_probability  optimal_probability  optimal_batch_1024_probability  required_lift_for_80pct_batch
    a3_dev_n4        development uniform_encoded_domain_wall_y              0.009766             0.000977                        0.632300                       1.608174
    a3_dev_n4        development  random_assignment_y_inferred              0.222222             0.024691                        1.000000                       0.063605
 a3_test_n4_1               test uniform_encoded_domain_wall_y              0.002441             0.000488                        0.393543                       3.216348
 a3_test_n4_1               test  random_assignment_y_inferred              0.049383             0.012346                        0.999997                       0.127209
 a3_test_n4_2               test uniform_encoded_domain_wall_y              0.004883             0.000977                        0.632300                  

## Bước5 — Đo hai solver placement cổ điển trong phiên hiện tại

Vét cạn duyệt đủ81/243 placements và trả nghiệm tốt nhất; random duyệt1024 lần,
không sửa overload. Solver không nhận đáp án optimum. Mỗi phép đo gồm input
validation, lựa chọn, kiểm tra và tạo output; loại imports/calibration/exact-label
preparation. Đây là warm-process timing, không đồng nhất với toàn thời gian notebook.
FF/BFD/MILP không có trong gói pilot này.

In [5]:
CLASSICAL_TIMINGS = []
for name, instance in CONTEXT['instances'].items():
    print('Đo placement controls:', name, flush=True)
    CLASSICAL_TIMINGS.extend(measure_assignment(instance))
save_json(FOLDER / 'classical_timings.json', CLASSICAL_TIMINGS)
table = pd.DataFrame(CLASSICAL_TIMINGS)
table['median_ms'] = 1000 * table['seconds_median']
display(table[['instance_id','method','median_ms','seconds_min','seconds_max','calls_per_block']])

Đo placement controls: a3_dev_n4
Đo placement controls: a3_test_n4_1
Đo placement controls: a3_test_n4_2
Đo placement controls: a3_test_n4_3
Đo placement controls: a3_dev_n5
Đo placement controls: a3_test_n5_1
Đo placement controls: a3_test_n5_2
Đo placement controls: a3_test_n5_3
Đo placement controls: a3_one_host
Đo placement controls: a3_infeasible
  instance_id            method  median_ms  seconds_min  seconds_max  calls_per_block
    a3_dev_n4 exact_enumeration   0.104353     0.000100     0.000109              256
    a3_dev_n4 random_assignment   2.387983     0.002316     0.004033                8
 a3_test_n4_1 exact_enumeration   0.154980     0.000128     0.000176              256
 a3_test_n4_1 random_assignment   2.238681     0.002147     0.002276                8
 a3_test_n4_2 exact_enumeration   0.106243     0.000098     0.000133              256
 a3_test_n4_2 random_assignment   2.279921     0.002211     0.004136                8
 a3_test_n4_3 exact_enumeration   0.104345  

## Bước6 — Probe tài nguyên tại cả ba độ sâu

Chạy3 điểm có mọi layer hoạt động và một batch cuối độc lập cho mỗi instance/p.
Passed nghĩa là các circuit đã chạy qua guard; maxiter_seconds_estimate là ngoại
suy bảo thủ, không bảo đảm whole training. Guards kiểm tra ở ranh giới evaluation,
không ngắt giữa native allocation. Hai dev lớn nhất23/25q; controls probe p1 riêng.
Nếu bị chặn, vẫn có thể chạy bước8–9 để xuất bằng chứng partial.

In [6]:
CAMPAIGN = run_phase(FOLDER, CONTEXT, 'probe', RUN_INSTANCES, RETRY_FAILED)
probes = [a for a in CAMPAIGN['attempts'] if a['kind'] == 'probe']
display(pd.DataFrame(probes)[['instance_id','reps','status','report']] if probes else pd.DataFrame())
for a in probes:
    result = json.loads((FOLDER / a['report']).read_text(encoding='utf-8'))['result']
    print(a['instance_id'], a['reps'], 'maxiter projection:', result.get('maxiter_seconds_estimate'),
          'within per-trial limit:', result.get('estimate_within_per_trial_limit'))
export_pilot(FOLDER, CONTEXT)

probe: a3_dev_n4 p=1 seed=190
passed
probe: a3_dev_n4 p=2 seed=190
passed
probe: a3_dev_n4 p=3 seed=190
passed
probe: a3_dev_n5 p=1 seed=190
passed
probe: a3_dev_n5 p=2 seed=190
passed
probe: a3_dev_n5 p=3 seed=190
passed
probe: a3_one_host p=1 seed=192
passed
probe: a3_infeasible p=1 seed=192
passed
  instance_id  reps status                                report
    a3_dev_n4     1 passed attempts/0001_probe_03db9f93b9d1.json
    a3_dev_n4     2 passed attempts/0002_probe_c56cd3f0b358.json
    a3_dev_n4     3 passed attempts/0003_probe_55e2f2742f32.json
    a3_dev_n5     1 passed attempts/0004_probe_05c351dcafaf.json
    a3_dev_n5     2 passed attempts/0005_probe_c4fbced6ed99.json
    a3_dev_n5     3 passed attempts/0006_probe_d4a4bd9d3b59.json
  a3_one_host     1 passed attempts/0007_probe_24b52dc6c082.json
a3_infeasible     1 passed attempts/0008_probe_eb4700dcb74f.json
a3_dev_n4 1 maxiter projection: 24.540987684000356 within per-trial limit: True
a3_dev_n4 2 maxiter projection: 3

## Bước7 — Development pilot và functional smokes

Hai dev × p1/2/3 ×2seed=12 lượt; thêm2control p1/cap35. Cùng khởi tạo ở mọi seed,
đây không phải multistart. Mean-energy/X/COBYLA cap200, train/final1024shots;
mọi extra sampler batch của Qiskit được tính. Output best-valid batch cuối,
không sửa bit hoặc y. Stop do COBYLA không chứng nhận tối ưu toàn cục.

In [7]:
passed = {(a['instance_id'], a['reps']) for a in CAMPAIGN['attempts']
          if a['kind'] == 'probe' and a['status'] == 'passed'}
needed = {(t['instance_id'],t['config']['reps']) for t in CAMPAIGN['planned_runs']
          if t['instance_id'] in RUN_INSTANCES}
if needed <= passed:
    CAMPAIGN = run_phase(FOLDER, CONTEXT, 'trial', RUN_INSTANCES, RETRY_FAILED, progress=True)
else:
    print('Chưa đủ probe PASS: bỏ qua training, giữ trạng thái và xuất partial.')

trial: a3_dev_n4 p=1 seed=190
  eval 1: E=628.906250, 1.2s
completed
trial: a3_dev_n4 p=1 seed=191
  eval 1: E=590.468750, 1.0s
completed
trial: a3_dev_n4 p=2 seed=190
  eval 1: E=628.906250, 1.9s
completed
trial: a3_dev_n4 p=2 seed=191
  eval 1: E=590.468750, 1.5s
completed
trial: a3_dev_n4 p=3 seed=190
  eval 1: E=628.906250, 2.1s
completed
trial: a3_dev_n4 p=3 seed=191
  eval 1: E=590.468750, 1.8s
completed
trial: a3_dev_n5 p=1 seed=190
  eval 1: E=559.109375, 4.8s
completed
trial: a3_dev_n5 p=1 seed=191
  eval 1: E=575.109375, 5.2s
completed
trial: a3_dev_n5 p=2 seed=190
  eval 1: E=559.109375, 6.7s
completed
trial: a3_dev_n5 p=2 seed=191
  eval 1: E=575.109375, 7.3s
completed
trial: a3_dev_n5 p=3 seed=190
  eval 1: E=559.109375, 8.8s
completed
trial: a3_dev_n5 p=3 seed=191
  eval 1: E=575.109375, 8.7s
completed
trial: a3_one_host p=1 seed=192
  eval 1: E=340.468750, 0.3s
completed
trial: a3_infeasible p=1 seed=192
  eval 1: E=1347.312500, 1.4s
completed


## Bước8 — Đọc chất lượng và chi phí pilot

P_opt là tỷ lệ final shots tối ưu; found_optimal là có ít nhất một mẫu tối ưu trong
một lượt. Không biến pooled P_opt thành TTS99 hoặc dùng2seed để chọn p thắng.
Các đường là development only, error bars SD giữa tối đa2seed, không phải CI
theo instance. Test chưa chạy. Tỷ lệ0 và mọi failure phải được giữ trong bảng.

In [8]:
CAMPAIGN = json.loads((FOLDER / 'campaign.json').read_text(encoding='utf-8'))
verify_pilot(FOLDER, CAMPAIGN, CONTEXT)
ROWS = result_rows(FOLDER, CAMPAIGN, CONTEXT)
quality = pd.DataFrame(ROWS)
display(quality[['instance_id','role','p','seed','status','found_valid','found_optimal',
    'feasible_probability','optimal_probability','invalid_domain_probability',
    'optimal_lift_encoded','optimal_lift_structured_random','total_seconds','total_algorithm_shots']])
completed = quality[(quality['role']=='development') & (quality['status']=='completed')]
fig, axes = plt.subplots(1,2,figsize=(11,4.3))
for ax,name in zip(axes,['a3_dev_n4','a3_dev_n5']):
    part = completed[completed['instance_id']==name]
    if not part.empty:
        means = part.groupby('p')['optimal_probability'].mean()
        sd = part.groupby('p')['optimal_probability'].std().fillna(0)
        ax.errorbar(means.index,100*means,yerr=100*sd,marker='o',capsize=4,label='QAOA development')
    for method,label,style in [('uniform_encoded_domain_wall_y','Uniform encoded',':'),
                               ('random_assignment_y_inferred','Random placement','--')]:
        q = next(c['optimal_probability'] for c in CONTROLS if c['instance_id']==name and c['method']==method)
        ax.axhline(100*q,linestyle=style,label=label)
    ax.set(title=name,xlabel='p',ylabel='Optimal final shots (%)',xticks=[1,2,3])
    ax.set_ylim(bottom=0);ax.grid(alpha=.2);ax.legend(fontsize=8)
fig.suptitle('Development pilot only — SD across seeds; test not run')
fig.tight_layout();fig.savefig(FOLDER/'pilot_quality.png',dpi=150);plt.show()

  instance_id               role  p  seed    status  found_valid found_optimal  feasible_probability  optimal_probability  invalid_domain_probability  optimal_lift_encoded  optimal_lift_structured_random  total_seconds  total_algorithm_shots
    a3_dev_n4        development  1   190 completed        False         False              0.000000                  0.0                    0.625000                   0.0                             0.0      10.527792                    640
    a3_dev_n4        development  1   191 completed        False         False              0.000000                  0.0                    0.750000                   0.0                             0.0      10.756161                    640
    a3_dev_n4        development  2   190 completed        False         False              0.000000                  0.0                    0.625000                   0.0                             0.0      15.681832                    640
    a3_dev_n4        development

## Bước9 — Xuất ZIP và checkpoint chi phí

Gửi ZIP dù partial. Save notebook rồi gửi kèm. Projection90trials dùng median
dev theo size/p và15test trials/group; chưa tính chi phí preparation của campaign
mới, khác cost structure hoặc failures. Không tự mở Notebook20 khi projection nhỏ.
Chỉ sau review pilot Windows mới khóa budget/phiên bản test. Bước này không đóng
Phase4 và không công bố lợi thế tốc độ.

In [9]:
ARCHIVE, ROWS, ASSESSMENT = export_pilot(FOLDER, CONTEXT)
print(json.dumps(ASSESSMENT, ensure_ascii=False, indent=2))
if ASSESSMENT['projected_90_trial_seconds'] is not None:
    print('Projection90trial:', round(ASSESSMENT['projected_90_trial_seconds']/3600,2), 'giờ; chưa phải dự báo bảo đảm.')
print('Gửi ZIP:', ARCHIVE)
print('Gửi kèm notebook19 sau Save. Giữ RESUME_DIR nếu pilot còn pending.')

{
  "status": "pilot_complete",
  "completed": 14,
  "planned": 14,
  "projected_90_trial_seconds": 4164.42650381249,
  "within_six_hours_by_projection": true,
  "projection_scope": "15 test trials per size/p times development median; excludes future preparation and failures; cost structure may differ; not a runtime guarantee",
  "depth_selection": "none",
  "test_campaign_started": false
}
Projection90trial: 1.16 giờ; chưa phải dự báo bảo đảm.
Gửi ZIP: /workspace/scratch/5e8f9a5e7776/nt409-task-placement/data/reference_results/phase4k_three_host_pilot/pilot_20261012T032855209810Z/pilot_20261012T032855209810Z.zip
Gửi kèm notebook19 sau Save. Giữ RESUME_DIR nếu pilot còn pending.
